# Grid & Matrix Traversal

A 2D grid is a graph in disguise: each cell is a **node**, and the moves to its neighbors are the **edges**. Once you see it that way, "count the islands," "how long until everything rots," and "shortest path through a maze" are all just **BFS/DFS** on an implicit graph — no adjacency list needed, because the neighbors are computed from coordinates. This notebook collects the reusable grid machinery: the directions list, the bounds check, the visited marking, and the multi-source twist.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **The template** — flood fill & counting regions
3. **Multi-source BFS** — simultaneous expansion
4. **In-place transforms** — rotate, spiral, boundary walk
5. **When to use & cheat-sheet**

## 1. The signal — a grid with moves to adjacent cells

Reach for grid traversal when the input is a **2D grid** and the problem is about reaching **adjacent cells** — stepping up/down/left/right (4-directional) or also diagonally (8-directional):

- "count the **islands** / connected regions / enclosed areas"
- "**flood fill** this region with a new color"
- "**shortest path** from A to B through open cells" (a maze)
- "how many steps until every cell is infected / rotten / flooded?"

Every one of these is **BFS or DFS on an implicit graph** (see the **graphs** notebook and **BFS/DFS**). The only thing that differs from a normal graph search: you never build an adjacency list. A cell's neighbors are just its coordinates plus each move in a small **directions list**, filtered by an **in-bounds** check. Mark cells **visited** as you go so you don't loop forever.

## 2. The template — flood fill & counting regions

Three pieces show up in *every* grid search:

1. a **directions list** — the offsets to each neighbor,
2. an **in-bounds** check — reject coordinates off the edge,
3. **visited** marking — a `set`, or flip the cell in place.

```
DIRS4 = [(1, 0), (-1, 0), (0, 1), (0, -1)]      # down, up, right, left
DIRS8 = DIRS4 + [(1, 1), (1, -1), (-1, 1), (-1, -1)]   # add diagonals

for dr, dc in DIRS4:
    nr, nc = r + dr, c + dc
    if 0 <= nr < R and 0 <= nc < C and not visited[nr][nc]:
        ...                                      # neighbor is valid -> recurse / enqueue
```

**Flood fill** is the kernel: from a start cell, visit every cell reachable through same-value neighbors. Here it is **both ways** — recursive DFS, and iterative BFS with a `deque` (the iterative form avoids Python's recursion limit on big grids):

In [1]:
from collections import deque

DIRS4 = [(1, 0), (-1, 0), (0, 1), (0, -1)]       # down, up, right, left


def flood_dfs(grid, r, c, seen):
    """Recursive DFS: mark this cell, recurse into same-value neighbors."""
    R, C = len(grid), len(grid[0])
    seen.add((r, c))
    for dr, dc in DIRS4:
        nr, nc = r + dr, c + dc
        if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] == 1 and (nr, nc) not in seen:
            flood_dfs(grid, nr, nc, seen)


def flood_bfs(grid, r, c, seen):
    """Iterative BFS: a deque queue instead of the call stack."""
    R, C = len(grid), len(grid[0])
    q = deque([(r, c)])
    seen.add((r, c))
    while q:
        cr, cc = q.popleft()
        for dr, dc in DIRS4:
            nr, nc = cr + dr, cc + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] == 1 and (nr, nc) not in seen:
                seen.add((nr, nc))               # mark on ENQUEUE, not on dequeue
                q.append((nr, nc))


grid = [
    [1, 1, 0, 0, 1],
    [1, 0, 0, 1, 1],
    [0, 0, 1, 0, 0],
    [0, 1, 1, 0, 1],
]
s1, s2 = set(), set()
flood_dfs(grid, 0, 0, s1)
flood_bfs(grid, 0, 0, s2)
print("DFS reached from (0,0):", sorted(s1))
print("BFS reached from (0,0):", sorted(s2))
assert s1 == s2 == {(0, 0), (0, 1), (1, 0)}      # same connected region, both ways

DFS reached from (0,0): [(0, 0), (0, 1), (1, 0)]
BFS reached from (0,0): [(0, 0), (0, 1), (1, 0)]


**Number of islands** — the canonical region count. Sweep every cell; each time you hit an unvisited land cell, that's a new region, so bump the counter and flood-fill it away. The visited set guarantees each cell is touched once, so the whole grid is **O(rows·cols)**:

In [2]:
def count_regions(grid):
    R, C = len(grid), len(grid[0])
    seen = set()
    count = 0
    for r in range(R):
        for c in range(C):
            if grid[r][c] == 1 and (r, c) not in seen:
                count += 1                       # a new island
                flood_bfs(grid, r, c, seen)      # consume the whole region
    return count


print("number of islands:", count_regions(grid))
assert count_regions(grid) == 4

number of islands: 4


Marking visited **in place** (overwrite land `1` → water `0`) saves the `set` entirely — the grid *is* the visited map. Use it when you're allowed to mutate the input; otherwise copy the grid or keep the set. (Switching `DIRS4` → `DIRS8` here would merge diagonally-touching islands.)

In [3]:
def count_regions_inplace(grid):
    grid = [row[:] for row in grid]              # don't clobber the caller's grid
    R, C = len(grid), len(grid[0])

    def sink(r, c):                              # DFS that zeroes the region
        if not (0 <= r < R and 0 <= c < C) or grid[r][c] == 0:
            return
        grid[r][c] = 0                           # "visited" = turned to water
        for dr, dc in DIRS4:
            sink(r + dr, c + dc)

    count = 0
    for r in range(R):
        for c in range(C):
            if grid[r][c] == 1:
                count += 1
                sink(r, c)
    return count


print("number of islands (in-place):", count_regions_inplace(grid))
assert count_regions_inplace(grid) == 4

number of islands (in-place): 4


## 3. Multi-source BFS — simultaneous expansion

When several sources spread **at the same time** — multiple fires, several infected cells, every gate in a maze — don't BFS from each one separately. Seed the queue with **all sources at once** at distance 0, then run one ordinary BFS. Because BFS expands in concentric rings, every cell is first reached by its **nearest** source, and the ring number is the time/distance. Still a single **O(rows·cols)** pass.

**Rotting oranges:** `2` = rotten, `1` = fresh, `0` = empty. Each minute, every rotten cell rots its 4-neighbors. How many minutes until none stay fresh? Seed every rotten orange, sweep outward, and the last ring's time is the answer:

In [4]:
def time_to_rot(grid):
    R, C = len(grid), len(grid[0])
    q = deque()
    fresh = 0
    for r in range(R):
        for c in range(C):
            if grid[r][c] == 2:
                q.append((r, c, 0))              # ALL sources enter at time 0
            elif grid[r][c] == 1:
                fresh += 1

    grid = [row[:] for row in grid]
    minutes = 0
    while q:
        r, c, t = q.popleft()
        minutes = max(minutes, t)
        for dr, dc in DIRS4:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] == 1:
                grid[nr][nc] = 2                 # rot it now (acts as visited mark)
                fresh -= 1
                q.append((nr, nc, t + 1))
    return minutes if fresh == 0 else -1         # -1: some fresh cell unreachable


oranges = [
    [2, 1, 1],
    [1, 1, 0],
    [0, 1, 1],
]
print("minutes to rot all:", time_to_rot(oranges))
assert time_to_rot(oranges) == 4
assert time_to_rot([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1   # bottom-left is stranded

minutes to rot all: 4


The same engine yields **distance to the nearest source** for every cell at once. Seed all sources at 0; the ring each cell lands in is its distance. Here two sources at opposite corners of a 3×4 grid — every cell gets the Manhattan-style nearest distance in one sweep:

In [5]:
def nearest_distance(R, C, sources):
    dist = [[-1] * C for _ in range(R)]
    q = deque()
    for r, c in sources:
        dist[r][c] = 0                           # multi-source seed
        q.append((r, c))
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS4:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1    # first visit = shortest (BFS rings)
                q.append((nr, nc))
    return dist


dist = nearest_distance(3, 4, sources=[(0, 0), (2, 3)])
for row in dist:
    print(row)
assert dist == [[0, 1, 2, 2], [1, 2, 2, 1], [2, 2, 1, 0]]

[0, 1, 2, 2]
[1, 2, 2, 1]
[2, 2, 1, 0]


## 4. In-place transforms — rotate, spiral, boundary walk

Not every grid problem is a search; some just **reshape** the matrix. Two idioms cover most of them.

**Rotate 90° clockwise = transpose, then reverse each row.** `zip(*m[::-1])` does it in one expression: `m[::-1]` flips the row order, and `zip(*...)` transposes — together that's a clockwise quarter-turn. (Reverse the two steps for counter-clockwise.)

In [6]:
def rotate_cw(m):
    return [list(row) for row in zip(*m[::-1])]  # reverse rows, then transpose


M = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]
rotated = rotate_cw(M)
for row in rotated:
    print(row)
assert rotated == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]   # top row was the left column
assert rotate_cw(rotate_cw(rotate_cw(rotate_cw(M)))) == M   # four turns = identity

[7, 4, 1]
[8, 5, 2]
[9, 6, 3]


**Spiral-order traversal** is the **boundary-walking** idiom: keep four bounds (`top, bottom, left, right`), walk the current top row, right column, bottom row, left column, then shrink the box inward and repeat. The two `if` guards stop a thin (single row/column) remainder from being walked twice:

In [7]:
def spiral_order(m):
    if not m:
        return []
    res = []
    top, bottom = 0, len(m) - 1
    left, right = 0, len(m[0]) - 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):         # top row, left -> right
            res.append(m[top][c])
        top += 1
        for r in range(top, bottom + 1):         # right column, top -> bottom
            res.append(m[r][right])
        right -= 1
        if top <= bottom:                        # bottom row, right -> left
            for c in range(right, left - 1, -1):
                res.append(m[bottom][c])
            bottom -= 1
        if left <= right:                        # left column, bottom -> top
            for r in range(bottom, top - 1, -1):
                res.append(m[r][left])
            left += 1
    return res


print("spiral 3x3:", spiral_order(M))
print("spiral 3x4:", spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))
assert spiral_order(M) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == \
    [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]

spiral 3x3: [1, 2, 3, 6, 9, 8, 7, 4, 5]
spiral 3x4: [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


**Two coordinate tricks** keep grid code short:

- **Complex numbers as coordinates.** Store a cell as `r + c*1j`; then a move is just **complex addition**, and the four unit moves are `(1, -1, 1j, -1j)`. No tuple unpacking, no separate bounds variables for the offset.
- **`itertools.product`** iterates every cell `(r, c)` in one flat loop — handy for seeding or scanning the whole grid.

In [8]:
import itertools

MOVES = (1, -1, 1j, -1j)                          # down/up + right/left as complex steps


def neighbors(z):
    return [z + d for d in MOVES]


print("neighbors of cell (2+3j):", neighbors(2 + 3j))
assert neighbors(2 + 3j) == [3 + 3j, 1 + 3j, 2 + 4j, 2 + 2j]

# itertools.product = a flat double loop over all (row, col) cells
all_cells = list(itertools.product(range(2), range(3)))
print("all cells of a 2x3 grid:", all_cells)
assert all_cells == [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]

neighbors of cell (2+3j): [(3+3j), (1+3j), (2+4j), (2+2j)]
all cells of a 2x3 grid: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]


## 5. When to use & cheat-sheet

| The grid problem is… | Reach for… |
|---|---|
| count regions / flood fill / enclosed areas | DFS or BFS from each unvisited cell |
| shortest path / fewest steps (unweighted) | **BFS** — first visit is the shortest |
| spread from many starts at once | **multi-source BFS** — seed all sources at distance 0 |
| weighted moves (costs differ) | Dijkstra / 0-1 BFS (see the **graphs** notebook) |
| rotate / spiral / reshape | boundary walking, `zip(*m[::-1])` |

**Python toolkit:** `collections.deque` is the BFS queue — O(1) `popleft`, unlike a list. Keep a **directions list** (`DIRS4` / `DIRS8`) and a one-line bounds check `0 <= nr < R and 0 <= nc < C`. **Mark visited in place** (overwrite the cell) when you may mutate the grid — it's faster and lighter than a `set`; use a `set` or a parallel `dist` matrix when you must preserve the input. `itertools.product` flattens the cell sweep; **complex numbers** turn a move into one addition.

**Two things to nail down:**
- **4- vs 8-connectivity** — do diagonal cells count as adjacent? It changes island counts and reachability. Pick `DIRS4` or `DIRS8` deliberately.
- **Why BFS gives shortest unweighted paths:** BFS expands the frontier in concentric rings, so the first time it reaches a cell it has used the fewest steps possible. DFS does *not* — it can reach a cell by a long winding route first. (Both are fine for *connectivity*; only BFS gives *distance*. See the **graphs** notebook.) Mark on **enqueue**, not dequeue, or a cell can land in the queue twice.

| Task | Approach | Time | Space |
|---|---|:---:|:---:|
| Flood fill / count regions | DFS or BFS | O(R·C) | O(R·C) |
| Shortest path (unweighted) | BFS | O(R·C) | O(R·C) |
| Distance from many sources | multi-source BFS | O(R·C) | O(R·C) |
| Rotate 90° / spiral order | one pass | O(R·C) | O(R·C) |

<sub>Every cell is visited a constant number of times and has at most 4 (or 8) neighbors, so a grid search is linear in the number of cells — the edge count is just a constant factor on the nodes.</sub>

---

A grid is the friendliest graph there is: the nodes are laid out for you and the edges are a `for` loop over four offsets. Master the directions-list + bounds-check + visited-mark skeleton once, and **number of islands**, **rotting oranges**, **maze shortest path**, and **flood fill** all become the same short program — the **BFS/DFS** from the **graphs** notebook, with the adjacency list left implicit.